# ResNet34 auf FER-2013 trainieren (Colab)

Dieses Notebook trainiert das ResNet34 aus dem Repo neu, diesmal mit sauberer Trennung:

| Teil | Woher | Wofür |
|---|---|---|
| Training | 90 % von `train/` | Gewichte lernen |
| Validierung | 10 % von `train/` (stratifiziert, Seed 42) | bestes Modell auswählen, Early Stopping |
| Test | `test/` | **nur einmal ganz am Ende** |

Checkpoints landen nach jeder Epoche in Google Drive. Wenn Colab die Verbindung trennt: Laufzeit neu verbinden, alle Zellen bis einschließlich **4. Training** nochmal ausführen. Das Training sollte dann bei der letzten fertigen Epoche weitermachen (noch nicht praktisch getestet, siehe Schritt 4).

**Vorher:** *Laufzeit → Laufzeittyp ändern → T4 GPU* (oder besser, falls verfügbar).

## 1. GPU prüfen

In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Keine GPU! Laufzeit → Laufzeittyp ändern → GPU auswählen"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Google Drive, Repo und Pakete

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/emotion-recognition"   # hier landen Checkpoints, Gewichte, Ergebnisse
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Speicherort:", DRIVE_DIR)

In [ ]:
REPO_DIR = "/content/emotion-recognition"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/FirasAlhashash/emotion-recognition.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
# Colab bringt torch, torchvision, scikit-learn und matplotlib schon mit (CUDA-Builds).
# Die Versionen aus requirements.txt sind für die lokale CPU-Umgebung, deshalb hier NICHT installieren,
# sonst wird die GPU-Version von torch überschrieben. Stattdessen die tatsächlich benutzten Versionen festhalten.
import sys, platform, torchvision, sklearn, numpy, matplotlib, PIL

versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "torchvision": torchvision.__version__,
    "scikit-learn": sklearn.__version__,
    "numpy": numpy.__version__,
    "matplotlib": matplotlib.__version__,
    "pillow": PIL.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
for k, v in versions.items():
    print(f"{k:13s} {v}")

os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
with open(f"{DRIVE_DIR}/results/colab_versions.txt", "w") as f:
    f.writelines(f"{k}=={v}\n" for k, v in versions.items())

## 3. Datensatz laden

Kaggle-Version von FER-2013 mit einem Ordner pro Emotion ([ananthu017/emotion-detection-fer](https://www.kaggle.com/datasets/ananthu017/emotion-detection-fer)). Die Daten kommen nach `/content/data` (lokale Colab-Platte, viel schneller als Drive).

Falls der Download ohne Login nicht klappt: `kaggle.json` hochladen und `!kaggle datasets download -d ananthu017/emotion-detection-fer -p /content` benutzen.

In [ ]:
DATA_DIR = "/content/data"

if not os.path.exists(f"{DATA_DIR}/test"):
    !curl -sL -o /content/fer.zip https://www.kaggle.com/api/v1/datasets/download/ananthu017/emotion-detection-fer
    !unzip -q -o /content/fer.zip -d {DATA_DIR}

# Kontrolle: muss 28709 / 7178 Bilder ergeben
for split in ["train", "test"]:
    counts = {c: len(os.listdir(f"{DATA_DIR}/{split}/{c}")) for c in sorted(os.listdir(f"{DATA_DIR}/{split}"))}
    print(split, sum(counts.values()), counts)

assert sum(len(os.listdir(f"{DATA_DIR}/train/{c}")) for c in os.listdir(f"{DATA_DIR}/train")) == 28709
assert sum(len(os.listdir(f"{DATA_DIR}/test/{c}")) for c in os.listdir(f"{DATA_DIR}/test")) == 7178

## 4. Training

Läuft mit `--resume`: gibt es schon einen Checkpoint in Drive, geht es dort weiter, sonst startet es neu. Diese Zelle kann man nach einer Unterbrechung nochmal starten.

**Achtung:** Das Fortsetzen ist noch nicht praktisch getestet. Nach einem Neustart prüfen, ob in der Ausgabe „Fortgesetzt nach Epoche …“ steht und die Epochenzählung an der richtigen Stelle weitergeht.

Das Testset wird hier nicht angefasst.

In [ ]:
CHECKPOINT = f"{DRIVE_DIR}/checkpoint.pth"
WEIGHTS = f"{DRIVE_DIR}/resnet34_fer2013.pth"
RESULTS = f"{DRIVE_DIR}/results"

!python train_resnet.py --data {DATA_DIR} --epochs 30 --batch-size 64 --lr 1e-4 --patience 7 --seed 42 --workers 2 --checkpoint {CHECKPOINT} --resume --weights-out {WEIGHTS} --out {RESULTS}

## 5. Trainingsverlauf

In [ ]:
import json
import matplotlib.pyplot as plt

with open(f"{RESULTS}/resnet34_training_history.json") as f:
    run = json.load(f)

h = run["history"]
epochs = [e["epoch"] for e in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, [e["train_loss"] for e in h])
ax1.set_title("Train-Loss")
ax1.set_xlabel("Epoche")
ax2.plot(epochs, [e["train_acc"] for e in h], label="Train")
ax2.plot(epochs, [e["val_acc"] for e in h], label="Validierung")
ax2.axvline(run["best_epoch"], color="gray", linestyle="--", label=f"bestes Modell (Epoche {run['best_epoch']})")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoche")
ax2.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS}/resnet34_training_curves.png", dpi=120)
plt.show()

print(f"Fertig: {run['finished']}   beste Val-Accuracy: {run['best_val_acc']:.4f} (Epoche {run['best_epoch']})")

## 6. Einmalige Auswertung auf dem Testset

Erst ausführen, wenn das Training **fertig** ist. Die Zelle verweigert sonst. Danach wird eine Markierung in Drive gesetzt, damit das Testset nicht aus Versehen mehrfach benutzt wird (z. B. um danach doch noch an Hyperparametern zu drehen).

In [ ]:
TEST_DONE = f"{RESULTS}/TEST_EVALUATED"

with open(f"{RESULTS}/resnet34_training_history.json") as f:
    run = json.load(f)

if not run["finished"]:
    print("Training ist noch nicht fertig, erst Schritt 4 zu Ende laufen lassen.")
elif os.path.exists(TEST_DONE):
    print("Testset wurde schon ausgewertet, Ergebnis:")
    print(open(f"{RESULTS}/resnet34_report.txt").read())
else:
    !python evaluate_resnet.py --data {DATA_DIR}/test --weights {WEIGHTS} --out {RESULTS}
    with open(TEST_DONE, "w") as f:
        f.write(f"best_epoch={run['best_epoch']}\nbest_val_acc={run['best_val_acc']:.4f}\n")

In [ ]:
from IPython.display import Image, display
display(Image(f"{RESULTS}/resnet34_confusion_matrix.png"))

## 7. Ergebnisse ins Repo übernehmen

Alles Wichtige liegt jetzt in Drive unter `MyDrive/emotion-recognition/`:

- `resnet34_fer2013.pth` → ins Repo nach `models/`
- `results/` (Report, Konfusionsmatrix, Trainingsverlauf, Kurven, Versionen) → ins Repo nach `results/`

`checkpoint.pth` braucht man danach nicht mehr, der ist nur zum Fortsetzen da.